<a href="https://colab.research.google.com/github/edwardoughton/GeoAI/blob/main/06_01_ggs590_geoai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 6 GeoAI: AI-supported automation for GIS workflows

This week focuses on workflow design, followed by using AI tools to accelerate implementation.

We will focus on a Land-Use, Land-Cover (LULC) example.

That means we need to sketch out a concept, to help us develop a baseline implementation for LULC prediction using:

- NAIP imagery from Microsoft Planetary Computer
- Labeled land-cover data for training (with controlled fallback)
- A Random Forest classifier as a transparent first model
- Tests and spatial validation to ensure trust



# Learning objectives

By the end of this class, students should be able to:
- Formulate a spatial Machine Learning (ML) workflow for LULC prediction
- Programmatically access NAIP and other labeled datasets from STAC
- Build a baseline Random Forest classifier model on raster extracted features
- Apply spatial holdout validation to reduce leakage risk
- Efficiently use AI coding assistants, but in a critical way (e.g., define your concept, generate, inspect, test, validate, refine, etc.)


# Class structure

1. Problem formulation
2. Workflow design
3. AI-assisted implementation
4. Model choice and critical reflection
5. Wrap-up


## Phase 1: Problem formulation

### 1. Conceptual overview

For a Random Forest, the approach is to train a model which can learn quantitative relationships like so:

$f$: $X$ → $Y$

Where:
- X = input feature space  
- Y = finite set of discrete class labels  

In other words, the model learns a function $𝑓$ that maps input features $𝑋$ to class labels $Y$.

The goal is to learn decision boundaries that separate classes in the feature space.

Here, we are aiming to predict discrete LULC categories (vegetation, water, concrete etc.), and not continuous quantitative values.





In a Random Forest, we want each pixel represented as a feature vector:

$$
\mathbf{X}_i = [\text{Band1}, \text{Band2}, \ldots, \ldots]
$$

Target variable:

$$
y_i = \text{Land Cover Class}
$$

Random Forest trains many decision trees using bootstrapped samples and random feature subsets, then combines predictions by majority vote.

Why this works well for land cover:
- Handles nonlinear class boundaries
- Robust to noisy observations
- Performs well with multi-spectral data
- Provides feature importance diagnostics


### 2. Data requirements
A. Multispectral imagery
- Examples: NASA Landsat, NAIP, etc.
- Typical bands: Blue, Green, Red, NIR, etc.

B. Ground-truth labels
- Field observations
- Interpreted/tagged reference data
- Existing land cover maps

C. Optional ancillary predictors
- DEM derivatives (elevation, slope, aspect)
- Spectral indices (NDVI, NDWI, NDBI)



### 3. Feature engineering
For each pixel, we combine raw bands and any derived variables in use (NDVI, NDWI, etc.).

Then, feature stacking yields our matrix $\mathbf{X}$, with one row per observation, and one column per feature (e.g., band):

$$
\mathbf{X} \in \mathbb{R}^{n \times p}
$$

Where:
- $n$ = number of sampled training pixels (rows)
- $p$ = number of features per pixel (columns)



### 4. Workflow pipeline
Preprocessing
- Harmonize CRS and resolution
- Mask invalid pixels
- Clip to Area Of Interest (AOI)

Training data preparation
- Extract pixel values under labels
- Check class balance
- Split into train/validation (prefer spatial split)

Trainning our Random Forest
- Typical range: `n_estimators=100-500`
- `max_features='sqrt'`
- `min_samples_leaf=1-5`
- `class_weight='balanced'` when imbalance exists



### 5. Evaluation strategies
We can assess our model using a range of metrics:
- Confusion matrix (counts)
- Row-normalized confusion matrix
- Precision, recall, F1
- Overall accuracy (interpret with caution)



### 7. Common issues
- Spatial leakage from random pixel splits (spatial leakage: when data from the same spatial area is in both training and testing datasets)
- Class imbalance dominating model behavior
- Overfitting to local texture/noise
- Mixed pixels and boundary uncertainty



### 8. Typical class schema examples
- Forest
- Grassland
- Agriculture
- Built-up areas
- Water
- Bare soil



### Exercise

Which predictors are physically meaningful for each class (e.g., for forest, grassland, built-up areas, water, bare soil, etc.)?

Write your answers here and then we will have a class discussion for each.



## Phase 2: Workflow design

Pipeline sketch:
1. Load an AOI boundary for case study/training area
2. Query NAIP imagery from Planetary Computer STAC
3. Query a label LULC dataset
4. Align raster grids and projections
5. Sample training pixels
6. Train Random Forest baseline
7. Validate with spatial holdout + confusion matrix
8. Predict and map outputs
9. Inspect errors and spatial coherence



## Workflow Concept Diagram

| Stage | Input | Operation | Output | Failure mode to check |
|---|---|---|---|---|
| A | AOI boundary | Define/validate AOI | AOI geometry | Wrong CRS / invalid polygon |
| B | AOI + STAC | Query NAIP | NAIP items | Empty query / wrong date |
| C | NAIP items | Cache + mosaic | AOI raster | Tile seams / nodata |
| D | NAIP raster | Feature extraction (R,G,B,NIR, NDVI optional) | Feature stack | Mis-scaled bands |
| E | Label source | Select labels | Label raster | Wrong class scheme |
| F | Labels + NAIP grid | Reproject/align | Aligned labels | Off-by-one / resampling artifacts |
| G | Features + labels | Sample pixels | Training set | Class imbalance / leakage |
| H | Training set | Spatial holdout split | Train/val sets | Adjacent leakage |
| I | Train set | Fit RF baseline | Model | Overfit / poor generalization |
| J | Val set | Evaluate | Metrics + confusion | OA misleading |
| K | AOI features | Predict | Full map | Edge artifacts / nodata bleed |
| L | Pred + labels | Error diagnostics | Error map | Spatially clustered failures |

### Exercise

Which steps are best AI-accelerated vs when we use human judgment?

Write your answers here and then we will have a class discussion for each.

## Dataset choice: strengths and tradeoffs

Choosing imagery and labels is a methodological decision, not just a data-access step.

### High-resolution aerial imagery (e.g., NAIP)
- Very high spatial detail (typically 1 m) and strong for local land-use mapping.
- Limited temporal frequency and some regional coverage constraints.

### Medium-resolution satellites (e.g., Landsat 9)
- Frequent revisit and broad/global coverage at 10-60 m.
- Coarser spatial resolutions (10 m+) do blur fine urban classes though.

### Practical guidance
- Match imagery resolution to your mapping objective.
- Check temporal consistency between imagery and labels.



## Setup

Run the install cell only if packages are missing.


In [1]:
# Optional installs (uncomment if needed)
!pip install planetary-computer pystac-client stackstac rioxarray rasterio xarray geopandas scikit-learn matplotlib seaborn


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.3/64.3 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 56.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.5/208.5 kB 12.6 MB/s eta 0:00:00
  Attempting uninstall: xarray
    Found existing installation: xarray 2025.12.0
    Uninstalling xarray-2025.12.0:
      Successfully uninstalled xarray-2025.12.0


In [2]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
from shapely.geometry import box

import xarray as xr
import rioxarray as rxr
from rioxarray.merge import merge_arrays
from urllib.parse import urlparse
from urllib.request import urlretrieve

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, balanced_accuracy_score

import pystac_client
import planetary_computer as pc


# AOI setup
This notebook downloads and caches the Fairfax City boundary (Virginia) in the `data/` folder,
then uses that boundary as the AOI for NAIP search and clipping.


In [3]:
# Settings you can edit in class
START_DATE = "2021-09-09"
END_DATE = "2021-09-10"
NAIP_COLLECTION = "naip"
NAIP_MAX_ITEMS = 100
NAIP_CACHE_DIR = Path("data/naip_cache")
FORCE_NAIP_DOWNLOAD = False
MAX_SAMPLES_PER_CLASS = 2500
RANDOM_STATE = 42

# Fairfax City AOI boundary settings
AOI_DOWNLOAD_URL = "https://raw.githubusercontent.com/edwardoughton/GeoAI/main/data/fairfax_city_boundary.gpkg"
AOI_PATH = Path("data/fairfax_city_boundary.gpkg")
FORCE_AOI_DOWNLOAD = False

# Optional class name map for plots/reports (edit for your chosen label dataset)
CLASS_LABELS = {
    # Example for ESA WorldCover-like classes:
    10: "Tree cover",
    20: "Shrubland",
    30: "Grassland",
    40: "Cropland",
    50: "Built-up",
    60: "Bare/sparse",
    70: "Snow/ice",
    80: "Permanent water",
    90: "Herbaceous wetland",
    95: "Mangroves",
    100: "Moss/lichen",
}



### Function Guide: Data Access Setup

- `get_pc_client()` connects to Planetary Computer with signed URL support.
- `download_fairfax_city_boundary()` downloads Fairfax City and writes it to `data/`.
- `load_aoi()` reads the AOI and can auto-download it from GitHub when missing.
- `search_naip_items()` queries NAIP tiles intersecting the AOI and date window.



In [4]:
def get_pc_client():
    """Create a signed Planetary Computer STAC client for authenticated asset access."""
    return pystac_client.Client.open(
        "https://planetarycomputer.microsoft.com/api/stac/v1",
        modifier=pc.sign_inplace,
    )


def load_aoi(aoi_path: Path, download_url: str | None = None, force_download: bool = False) -> gpd.GeoDataFrame:
    """Load AOI boundary data, downloading it from a URL when missing (or when forced)."""
    needs_download = force_download or (not aoi_path.exists())

    if needs_download:
        if not download_url:
            raise FileNotFoundError(f"AOI file not found and no download URL provided: {aoi_path}")
        aoi_path.parent.mkdir(parents=True, exist_ok=True)
        print(f"Downloading AOI to {aoi_path}...")
        urlretrieve(download_url, aoi_path)

    if not aoi_path.exists():
        raise FileNotFoundError(f"AOI file not found after download attempt: {aoi_path}")

    aoi = gpd.read_file(aoi_path)
    if aoi.crs is None:
        raise ValueError("AOI has no CRS")
    return aoi

def search_naip_items(client, aoi_gdf, start_date, end_date):
    """Search NAIP items intersecting the AOI within the configured date window."""
    aoi_wgs84 = aoi_gdf.to_crs(4326)
    geom = aoi_wgs84.geometry.union_all().__geo_interface__
    search = client.search(
        collections=[NAIP_COLLECTION],
        intersects=geom,
        datetime=f"{start_date}/{end_date}",
        limit=NAIP_MAX_ITEMS,
    )
    items = list(search.items())
    if not items:
        raise ValueError(
            f"No NAIP items found for AOI and date range {start_date} to {end_date}. "
            "Check AOI location, date window, and collection availability."
        )
    return items



### Function Guide: Collection Discovery

- `find_collections()` scans collection metadata for required keywords.
- This helps students discover candidate label sources before selecting one.


In [5]:
def find_collections(client, keywords):
    """Return STAC collections whose id/title/description contains all provided keywords."""
    rows = []
    for c in client.get_collections():
        text = " ".join([
            c.id or "",
            c.title or "",
            c.description or "",
        ]).lower()
        if all(k.lower() in text for k in keywords):
            rows.append({"id": c.id, "title": c.title})
    return pd.DataFrame(rows)


client = get_pc_client()
lulc_candidates = find_collections(client, ["land", "cover"])

print("General land-cover collection candidates:")
display(lulc_candidates.head(10))


General land-cover collection candidates:


,id,title
0,gnatsgo-tables,gNATSGO Soil Database - Tables
1,hgb,HGB: Harmonized Global Biomass for 2010
2,gnatsgo-rasters,gNATSGO Soil Database - Rasters
3,io-lulc-annual-v02,10m Annual Land Use Land Cover (9-class) V2
4,conus404,CONUS404
5,sentinel-1-rtc,Sentinel 1 Radiometrically Terrain Corrected (...
6,noaa-c-cap,C-CAP Regional Land Cover and Change
7,alos-fnf-mosaic,ALOS Forest/Non-Forest Annual Mosaic
8,chloris-biomass,Chloris Biomass
9,modis-11A2-061,MODIS Land Surface Temperature/Emissivity 8-Day


In [6]:
# Set this after inspecting candidates.
# Example fallback:
LABEL_COLLECTION = "esa-worldcover"
LABEL_ASSET_KEY_HINT = "map"  # change if your chosen collection uses another key


### Function Guide: Label Item Utilities

- `choose_asset_key()` picks the most likely raster asset key from a label item.
- `get_label_item()` retrieves a label item intersecting the AOI.


In [7]:
def choose_asset_key(item, hint=None):
    """Choose a raster asset key from a STAC item, using an optional preferred key hint."""
    keys = list(item.assets.keys())
    if hint and hint in keys:
        return hint

    geotiff_like = []
    for k, asset in item.assets.items():
        mt = (asset.media_type or "").lower()
        href = (asset.href or "").lower()
        if "tiff" in mt or href.endswith(".tif") or href.endswith(".tiff"):
            geotiff_like.append(k)

    if geotiff_like:
        return geotiff_like[0]
    return keys[0]


def get_label_item(client, aoi_gdf, collection_id):
    """Fetch a label item intersecting the AOI from the selected collection."""
    aoi_wgs84 = aoi_gdf.to_crs(4326)
    geom = aoi_wgs84.geometry.union_all().__geo_interface__
    search = client.search(
        collections=[collection_id],
        intersects=geom,
        limit=10,
    )
    items = list(search.items())
    if not items:
        raise ValueError(f"No label items found for collection '{collection_id}' in AOI")
    return items[0]


### Function Guide: NAIP Caching and Mosaic

- `_naip_local_path()` creates deterministic local cache filenames.
- `download_naip_assets()` downloads signed NAIP imagery once for reuse.
- `stack_naip_median()` clips/reprojects/merges tiles for AOI coverage.
- `load_label_raster()` opens the selected label raster asset.


In [8]:
def _naip_local_path(item, cache_dir: Path):
    """Build a deterministic local filename for a NAIP STAC item in the cache directory."""
    cache_dir.mkdir(parents=True, exist_ok=True)
    parsed = urlparse(item.assets["image"].href)
    base_name = Path(parsed.path).name or f"{item.id}.tif"
    safe_item_id = str(item.id).replace("/", "_").replace("\\", "_")
    return cache_dir / f"{safe_item_id}_{base_name}"


def download_naip_assets(items, cache_dir=Path("data/naip_cache"), force_download=False):
    """Download NAIP image assets to local cache and return local file paths."""
    local_paths = []
    for item in items:
        out_path = _naip_local_path(item, cache_dir)
        if force_download or not out_path.exists():
            signed_href = pc.sign(item.assets["image"].href)
            urlretrieve(signed_href, out_path)
        local_paths.append(out_path)
    return local_paths


def stack_naip_median(items, aoi_gdf, cache_dir=Path("data/naip_cache"), force_download=False):
    """Create an AOI-wide NAIP mosaic by caching, clipping, reprojection, and tile merge."""
    if not items:
        raise ValueError("No NAIP items found")

    local_paths = download_naip_assets(items, cache_dir=cache_dir, force_download=force_download)

    rasters = []
    for path in local_paths:
        da = rxr.open_rasterio(path, masked=True).astype("float32")  # (band, y, x)

        # Clip in source CRS first, then project to the common analysis CRS.
        aoi_src = aoi_gdf.to_crs(da.rio.crs)
        minx, miny, maxx, maxy = aoi_src.total_bounds

        try:
            da = da.rio.clip_box(minx=minx, miny=miny, maxx=maxx, maxy=maxy)
        except Exception:
            continue

        da = da.rio.reproject("EPSG:3857", resolution=1)
        rasters.append(da)

    if not rasters:
        raise ValueError("No NAIP rasters overlapped AOI after clipping")

    # Merge tiles to union coverage so the full AOI can be represented.
    naip_mosaic = merge_arrays(rasters, nodata=np.nan)
    return naip_mosaic


def load_label_raster(item, asset_hint=None):
    """Load a label raster from the chosen STAC item and return raster + selected asset key."""
    key = choose_asset_key(item, asset_hint)
    href = pc.sign(item.assets[key].href)
    lbl = rxr.open_rasterio(href, masked=True).squeeze(drop=True)
    return lbl, key



In [ ]:
aoi = load_aoi(AOI_PATH, download_url=AOI_DOWNLOAD_URL, force_download=FORCE_AOI_DOWNLOAD)
naip_items = search_naip_items(client, aoi, START_DATE, END_DATE)
print(f"NAIP items found: {len(naip_items)}")

naip_median = stack_naip_median(
    naip_items,
    aoi,
    cache_dir=NAIP_CACHE_DIR,
    force_download=FORCE_NAIP_DOWNLOAD,
)
print(naip_median)
print(f"Cached NAIP files: {len(list(NAIP_CACHE_DIR.glob('*.tif')))} in {NAIP_CACHE_DIR}")



NAIP items found: 2


### Stage A outputs: AOI and NAIP composite preview

At this stage we validate two fundamentals before modeling:
- The AOI geometry is where we expect.
- The NAIP composite has valid visual structure (not empty, shifted, or distorted).


In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 8))

# NAIP RGB quicklook
r = naip_median.sel(band=1).values
g = naip_median.sel(band=2).values
b = naip_median.sel(band=3).values
rgb = np.stack([r, g, b], axis=-1)

# Robust stretch for visualization
p2, p98 = np.nanpercentile(rgb, [2, 98])
rgb_vis = np.clip((rgb - p2) / (p98 - p2 + 1e-6), 0, 1)

extent = [float(naip_median.x.min()), float(naip_median.x.max()), float(naip_median.y.min()), float(naip_median.y.max())]
ax.imshow(rgb_vis, extent=extent, origin='upper')

# AOI boundary overlay
aoi_boundary = aoi.to_crs(3857).boundary
aoi_boundary.plot(ax=ax, color='yellow', linewidth=1.5)

from matplotlib.lines import Line2D
legend_elements = [Line2D([0], [0], color='yellow', lw=2, label='AOI boundary')]
ax.legend(handles=legend_elements, loc='lower right', frameon=True)

ax.set_title('NAIP median composite (RGB) with AOI overlay')
ax.set_axis_off()

plt.tight_layout()
plt.show()



In [ ]:
label_item = get_label_item(client, aoi, LABEL_COLLECTION)
labels_raw, label_asset_key = load_label_raster(label_item, LABEL_ASSET_KEY_HINT)

print(f"Using label collection: {LABEL_COLLECTION}")
print(f"Using label asset: {label_asset_key}")
print(labels_raw)


In [ ]:
# Reproject labels to the NAIP grid.
labels = labels_raw.rio.reproject_match(naip_median, resampling=0)

# Clip to AOI for a cleaner training area.
aoi_3857 = aoi.to_crs(3857)
labels = labels.rio.clip(aoi_3857.geometry, aoi_3857.crs, drop=True)
naip_clip = naip_median.rio.clip(aoi_3857.geometry, aoi_3857.crs, drop=True)

print("NAIP shape:", tuple(naip_clip.shape))
print("Label shape:", tuple(labels.shape))


### Stage B outputs: label data before and after alignment

This check confirms that labels are both:
- semantically sensible (class pattern looks plausible), and
- geometrically aligned to the NAIP analysis grid.


In [ ]:
# Fast preview: clip raw labels to AOI extent in source CRS, then downsample for plotting.
aoi_lbl = aoi.to_crs(labels_raw.rio.crs)
minx, miny, maxx, maxy = aoi_lbl.total_bounds
labels_raw_clip = labels_raw.rio.clip_box(minx=minx, miny=miny, maxx=maxx, maxy=maxy)

# Coarsen only for visualization speed (does not affect modeling data).
raw_plot = labels_raw_clip.coarsen(x=4, y=4, boundary="trim").max()
aligned_plot = labels.coarsen(x=4, y=4, boundary="trim").max()

# Build discrete class legend from aligned labels.
vals = aligned_plot.values
classes = sorted([int(v) for v in np.unique(vals[np.isfinite(vals)]) if int(v) > 0])
if not classes:
    classes = [0]

cmap = plt.cm.get_cmap('tab20', len(classes))
class_to_idx = {c: i for i, c in enumerate(classes)}

raw_vals = raw_plot.values
aligned_vals = aligned_plot.values
raw_idx = np.full(raw_vals.shape, np.nan, dtype='float32')
aligned_idx = np.full(aligned_vals.shape, np.nan, dtype='float32')

for c, idx in class_to_idx.items():
    raw_idx[np.isfinite(raw_vals) & (raw_vals.astype(int) == c)] = idx
    aligned_idx[np.isfinite(aligned_vals) & (aligned_vals.astype(int) == c)] = idx

fig, ax = plt.subplots(1, 2, figsize=(16, 5))

raw_im = ax[0].imshow(
    raw_idx,
    cmap=cmap,
    vmin=-0.5,
    vmax=len(classes)-0.5,
    extent=[float(raw_plot.x.min()), float(raw_plot.x.max()), float(raw_plot.y.min()), float(raw_plot.y.max())],
    origin='upper'
)
ax[0].set_title('Raw labels clipped to AOI (quicklook)')
ax[0].set_axis_off()

aligned_im = ax[1].imshow(
    aligned_idx,
    cmap=cmap,
    vmin=-0.5,
    vmax=len(classes)-0.5,
    extent=[float(aligned_plot.x.min()), float(aligned_plot.x.max()), float(aligned_plot.y.min()), float(aligned_plot.y.max())],
    origin='upper'
)
aoi_3857.boundary.plot(ax=ax[1], color='black', linewidth=1)
ax[1].set_title('Aligned labels (quicklook)')
ax[1].set_axis_off()

from matplotlib.patches import Patch
legend_handles = [
    Patch(facecolor=cmap(i), edgecolor='black', label=f"{c}: {CLASS_LABELS.get(c, f'Class {c}')}")
    for i, c in enumerate(classes)
]
fig.legend(
    handles=legend_handles,
    title='Land-use / land-cover classes',
    loc='lower center',
    bbox_to_anchor=(0.5, -0.08),
    ncol=min(6, len(legend_handles)),
    frameon=True,
)

plt.tight_layout(rect=(0, 0.08, 1, 1))
plt.show()





## Visual checkpoints before modeling

A reliable GeoAI workflow uses progressive diagnostics. The maps above are not optional cosmetics; they are validity checks that prevent silent spatial errors from propagating into the model stage.


## Phase 3: AI-assisted implementation with controls

You can use an AI assistant to generate/refactor the next functions, but enforce these requirements:
- Keep CRS alignment explicit
- Reproject labels to NAIP grid only once
- Mask nodata before sampling
- Use spatial holdout, not only random split
- Return metrics and confusion matrix


### Function Guide: Sampling and Spatial Split

- `build_training_table()` converts aligned rasters to tabular samples with class balancing.
- `spatial_holdout_split()` creates spatially disjoint train/test sets using x-index partitioning.


In [ ]:
def build_training_table(naip_da, labels_da, max_samples_per_class=2500, random_state=42):
    """Convert aligned raster bands and labels into a sampled tabular training dataframe."""
    bands, ny, nx = naip_da.shape

    X = naip_da.values.reshape(bands, ny * nx).T
    y = labels_da.values.reshape(ny * nx)

    # Remove missing labels and missing features.
    valid = np.isfinite(y) & np.all(np.isfinite(X), axis=1)
    X = X[valid]
    y = y[valid].astype(np.int32)

    if len(y) == 0:
        raise ValueError("No valid training pixels remain after removing missing values.")

    # Remove 0/no-data-like classes when present.
    keep = y > 0
    X = X[keep]
    y = y[keep]

    if len(y) == 0:
        raise ValueError("No labeled training pixels remain after removing nodata / class 0 values.")

    # Lightweight spatial keys for holdout and map diagnostics.
    yy, xx = np.indices((ny, nx))
    xx = xx.reshape(ny * nx)[valid][keep]
    yy = yy.reshape(ny * nx)[valid][keep]

    df = pd.DataFrame(X, columns=[f"band_{i+1}" for i in range(bands)])
    df["label"] = y
    df["x_index"] = xx
    df["y_index"] = yy

    if df.empty:
        raise ValueError("Training dataframe is empty after feature/label filtering.")

    # Cap samples per class to keep class time manageable.
    sampled = []
    rng = np.random.default_rng(random_state)
    for cls, g in df.groupby("label"):
        if g.empty:
            continue
        n = min(len(g), max_samples_per_class)
        idx = rng.choice(g.index.values, size=n, replace=False)
        sampled.append(df.loc[idx])

    if not sampled:
        raise ValueError("No class samples were available for model training.")

    out = pd.concat(sampled, ignore_index=True)
    return out


def spatial_holdout_split(df, holdout_quantile=0.8):
    """Split samples by x-coordinate quantile to enforce spatially disjoint train/test sets."""
    if df.empty:
        raise ValueError("Cannot split an empty training dataframe.")

    cutoff = df["x_index"].quantile(holdout_quantile)
    train_df = df[df["x_index"] <= cutoff].copy()
    test_df = df[df["x_index"] > cutoff].copy()

    if train_df.empty or test_df.empty:
        raise ValueError(
            "Spatial holdout produced an empty train or test set. "
            "Adjust sampling density, AOI size, or holdout_quantile."
        )

    return train_df, test_df, cutoff


In [ ]:
def assert_crs_match(a, b):
    """Assert that two raster-like objects share the same CRS."""
    assert str(a.rio.crs) == str(b.rio.crs), "CRS mismatch"


def assert_shapes_match(naip_da, labels_da):
    """Assert that feature and label rasters share identical y/x grid shape."""
    assert naip_da.shape[-2:] == labels_da.shape[-2:], "Grid shape mismatch"


def assert_no_nan(df, feature_cols):
    """Assert that selected feature columns contain only finite numeric values."""
    assert np.isfinite(df[feature_cols].to_numpy()).all(), "NaN/Inf in features"


def assert_spatial_disjoint(train_df, test_df):
    """Assert that train and test sets do not overlap in the x-index split dimension."""
    train_max = train_df["x_index"].max()
    test_min = test_df["x_index"].min()
    assert train_max < test_min, "Spatial holdout is not disjoint"

# Validation checks before training
assert_crs_match(naip_clip, labels)
assert_shapes_match(naip_clip, labels)

train_table = build_training_table(
    naip_clip,
    labels,
    max_samples_per_class=MAX_SAMPLES_PER_CLASS,
    random_state=RANDOM_STATE,
)

feature_cols = [c for c in train_table.columns if c.startswith("band_")]
assert_no_nan(train_table, feature_cols)

train_df, test_df, cutoff = spatial_holdout_split(train_table, holdout_quantile=0.8)
assert_spatial_disjoint(train_df, test_df)

print("Training rows:", len(train_df))
print("Testing rows:", len(test_df))
print("Classes in train:", sorted(train_df.label.unique())[:10], "...")


### Stage C outputs: training sample diagnostics

Before fitting, we inspect:
- class balance after sampling caps, and
- spatial separation between train/test holdout partitions.


In [ ]:
# Class distribution
class_counts = train_table['label'].value_counts().sort_index()
class_names = [CLASS_LABELS.get(int(c), f"Class {int(c)}") for c in class_counts.index]

fig, ax = plt.subplots(1, 2, figsize=(16, 5))
ax[0].bar(class_names, class_counts.values)
ax[0].set_title('Training samples per class')
ax[0].set_ylabel('Count')
ax[0].tick_params(axis='x', rotation=45)

# Spatial split diagnostics (sampled points)
xvals = naip_clip.x.values
yvals = naip_clip.y.values

train_plot = train_df.sample(min(2000, len(train_df)), random_state=RANDOM_STATE)
test_plot = test_df.sample(min(2000, len(test_df)), random_state=RANDOM_STATE)

train_x = xvals[train_plot['x_index'].to_numpy().astype(int)]
train_y = yvals[train_plot['y_index'].to_numpy().astype(int)]
test_x = xvals[test_plot['x_index'].to_numpy().astype(int)]
test_y = yvals[test_plot['y_index'].to_numpy().astype(int)]

ax[1].scatter(train_x, train_y, s=3, alpha=0.4, label='Train', c='tab:blue')
ax[1].scatter(test_x, test_y, s=3, alpha=0.4, label='Test', c='tab:orange')
aoi_3857.boundary.plot(ax=ax[1], color='black', linewidth=1)
ax[1].set_title('Spatial holdout split (sampled points)')
ax[1].legend(loc='lower right')
ax[1].set_axis_off()

plt.tight_layout()
plt.show()


### Why Random Forest

- We use Random Forest as a strong tabular baseline for pixel-wise classification.
- `n_estimators=250` stabilizes ensemble behavior without excessive runtime.
- `class_weight="balanced_subsample"` helps reduce bias toward dominant classes.
- This is the reference model before moving to higher-complexity methods.


In [ ]:
rf = RandomForestClassifier(
    n_estimators=250,
    max_depth=None,
    min_samples_leaf=1,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    class_weight="balanced_subsample",
)

X_train = train_df[feature_cols].to_numpy()
y_train = train_df["label"].to_numpy()
X_test = test_df[feature_cols].to_numpy()
y_test = test_df["label"].to_numpy()

rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("Balanced accuracy:", round(balanced_accuracy_score(y_test, y_pred), 4))

present_classes = sorted(np.unique(np.concatenate([y_test, y_pred])))
target_names = [CLASS_LABELS.get(int(c), f"Class {int(c)}") for c in present_classes]

print(
    classification_report(
        y_test,
        y_pred,
        labels=present_classes,
        target_names=target_names,
        zero_division=0,
    )
)



### Confusion Matrix (Raw Counts)

- This matrix shows the number of samples for each observed/predicted class pair.
- Diagonal cells are correct predictions; off-diagonal cells are misclassifications.
- Use this view to inspect absolute error volume by class.


In [ ]:
present_classes = sorted(np.unique(np.concatenate([y_test, y_pred])))
class_names = [CLASS_LABELS.get(int(c), f"Class {int(c)}") for c in present_classes]

cm = confusion_matrix(y_test, y_pred, labels=present_classes)
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    cm,
    cmap="Blues",
    ax=ax,
    xticklabels=class_names,
    yticklabels=class_names,
    annot=True,
    fmt="d",
)
ax.set_title("Confusion matrix (spatial holdout)")
ax.set_xlabel("Predicted")
ax.set_ylabel("Observed")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()




### Confusion Matrix (Row-Normalized)

- This view normalizes each row (true class) to proportions.
- Values approximate per-class recall distribution across predicted classes.
- It helps compare performance fairly when classes have different sample sizes.


In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

fig, ax = plt.subplots(figsize=(9, 7))
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    labels=present_classes,
    display_labels=class_names,
    normalize="true",  # row normalization
    cmap="Blues",
    values_format=".2f",
    xticks_rotation=45,
    ax=ax,
)
ax.set_title("Normalized confusion matrix (row-wise %)")
plt.tight_layout()
plt.show()



### Interpreting Feature Importance

Random Forest feature importance is computed from how much each feature reduces impurity across all trees.

How these results are produced:
- Each decision tree split chooses a feature (e.g., Red, Green, Blue, NIR).
- Splits that improve class separation contribute more impurity reduction.
- Importances are aggregated across all trees and normalized so they sum to 1.

How to interpret the chart:
- Higher importance means the model used that feature more often and/or more effectively.
- Importance is relative (ranking), not causal proof.
- If NIR is high, it often indicates strong vegetation/non-vegetation separation power.


In [ ]:
# Feature importance inspection
# Map model feature keys to human-readable remote-sensing names.
feature_name_map = {
    "band_1": "Red",
    "band_2": "Green",
    "band_3": "Blue",
    "band_4": "NIR",
    "ndvi": "NDVI",
}

pretty_feature_names = [feature_name_map.get(f, f) for f in feature_cols]
fi = pd.Series(rf.feature_importances_, index=pretty_feature_names).sort_values(ascending=False)

print("Feature importance scores (higher means the model relied more on that feature for splits):")
print(fi)

fig, ax = plt.subplots(figsize=(7, 3.5))
fi.plot(kind="bar", ax=ax, color="steelblue", edgecolor="black")
ax.set_title("Random Forest feature importance")
ax.set_ylabel("Importance")
ax.set_xlabel("Feature")
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()



### Predicting the Full AOI Map

- After validation, we apply the trained model to every valid AOI pixel.
- We preserve nodata as NaN and only predict on finite feature vectors.
- The resulting raster is the model-derived land-cover surface for mapping and diagnostics.


In [ ]:
# Predict full AOI map
bands, ny, nx = naip_clip.shape
X_full = naip_clip.values.reshape(bands, ny * nx).T
valid_full = np.all(np.isfinite(X_full), axis=1)

pred_full = np.full(X_full.shape[0], np.nan)
pred_full[valid_full] = rf.predict(X_full[valid_full])

pred_map = xr.DataArray(
    pred_full.reshape(ny, nx),
    coords={"y": naip_clip.y, "x": naip_clip.x},
    dims=("y", "x"),
).rio.write_crs(naip_clip.rio.crs)

# Discrete classes + legend
pred_vals = pred_map.values
pred_classes = sorted([int(v) for v in np.unique(pred_vals[np.isfinite(pred_vals)]) if int(v) > 0])
if not pred_classes:
    pred_classes = [0]

cmap = plt.cm.get_cmap('tab20', len(pred_classes))
class_to_idx = {c: i for i, c in enumerate(pred_classes)}
plot_idx = np.full(pred_vals.shape, np.nan, dtype='float32')
for c, idx in class_to_idx.items():
    plot_idx[np.isfinite(pred_vals) & (pred_vals.astype(int) == c)] = idx

fig, ax = plt.subplots(figsize=(8, 5))
ax.imshow(
    plot_idx,
    cmap=cmap,
    vmin=-0.5,
    vmax=len(pred_classes)-0.5,
    extent=[float(pred_map.x.min()), float(pred_map.x.max()), float(pred_map.y.min()), float(pred_map.y.max())],
    origin='upper'
)
aoi_3857.boundary.plot(ax=ax, color="black", linewidth=1)

from matplotlib.patches import Patch
legend_handles = [
    Patch(facecolor=cmap(i), edgecolor='black', label=f"{c}: {CLASS_LABELS.get(c, f'Class {c}')}")
    for i, c in enumerate(pred_classes)
]
fig.legend(
    handles=legend_handles,
    title='Predicted classes',
    loc='lower center',
    bbox_to_anchor=(0.5, -0.04),
    ncol=6,
    frameon=True,
)

ax.set_title("Predicted LULC classes (Random Forest baseline)")
ax.set_axis_off()
plt.tight_layout(rect=(0, 0.08, 1, 1))
plt.show()




### Stage D outputs: label vs prediction comparison

Final diagnostic:
- Left panel shows aligned reference labels.
- Right panel shows Random Forest predictions.

Use this to discuss where model errors cluster and what workflow component to improve next.


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 6))

labels.plot(ax=ax[0], cmap='tab20', add_colorbar=False)
aoi_3857.boundary.plot(ax=ax[0], color='black', linewidth=1)
ax[0].set_title('Reference labels (aligned)')
ax[0].set_axis_off()

pred_map.plot(ax=ax[1], cmap='tab20', add_colorbar=False)
aoi_3857.boundary.plot(ax=ax[1], color='black', linewidth=1)
ax[1].set_title('Model prediction map')
ax[1].set_axis_off()

plt.tight_layout()
plt.show()


### Exercise: Spatial trust check
- Compare predicted classes against labels in 2-3 sub-areas.
- Are errors spatially clustered?
- Which class pairs are most frequently confused?
- What would you change next: features, labels, sampling, or model?
